# M3GNet XAS large sweep (4 encoders x 8 UniversalXAS x 16 tuned)

This notebook trains 4 scratch M3GNet encoders, exports their features, trains
8 UniversalXAS head variants per encoder, then trains 16 tuned variants per
element from each encoder's best universal head. Selection uses the
validation split only. Test is evaluated exactly once at the end.

## Stages

1. **Stage 1:** 4 encoders, trained sequentially (up to 1000 epochs each).
   Each encoder exports its features once.
2. **Stage 2:** 8 UniversalXAS head variants per encoder, 4 parallel workers.
3. **Stage 3:** 16 tuned variants x 8 elements per encoder, 4 parallel
   workers, from the encoder's best universal head.

Expected wall time: several days on a single GPU box (4 encoder fits plus 32
universal and 512 tuned head jobs).

## Prerequisites

From the repository root, run:

```bash
bash tutorial_omnixas/download_omnixas_raw_data.sh
export OMNIXAS_DATA_ROOT="$HOME/OmniXAS_data"
```

The script downloads and extracts FEFF data by default. VASP download is not
needed for this FEFF pipeline.

The shell script requires `curl`, `md5sum`, and `tar`.

## Selection protocol (scientific invariants)

- Selection uses the validation split only.
- Best universal per encoder = argmax macro validation eta over the 8
  variants (macro eta = unweighted mean of the 8 per-element validation
  etas).
- Per-element tuned winner = argmax validation eta over the 16 variants.
- Test is evaluated exactly once, after all selection: (a) the 4 best
  universal heads and (b) the 4 x 8 tuned winner heads. Nothing is chosen
  using test metrics.
- Seeds are fixed at 42 for every variant on purpose: variants compare
  hyperparameters, not seed variance.
- Feature families are per-encoder: a head is never trained on one encoder's
  features and evaluated on another's.
- Provenance is recorded per run: spec, seed, source checkpoint sha256
  (tuned jobs), feature file sha256, and git sha.

## Dead ends avoided (git history)

- GELU/LeakyReLU activation sweep, GELU batch-size sweep, GELU
  normalization sweep: GELU rejected in `research/IMPROVEMENT_PLAN.md`;
  deleted in `834553aa`.
- Per-element head widths: `train_paper_models.py` deleted in `7adefbcb`.
- 128D weighted-shell encoder: `cu_feff_encoder_study.ipynb` deleted in
  `7adefbcb`. `enc_c_wide128` here is plain M3GNet feature width (128D
  node/edge features), not weighted shells.
- Joint E2E encoder+head training variants: `m3gnet_all8_feff_encoder.py` /
  `run_all8_feff_pipeline.py` deleted in `46636949`; `train_e2e_custom_*.py`
  deleted in `834553aa`.
- Plateau checked every 2 epochs with patience 8: old pipeline; replaced by
  ReduceLROnPlateau patience 16 checked every epoch in the current pipeline.
- VASP encoder experts: deleted in `834553aa`.

In [ ]:
import os
from pathlib import Path

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'tutorial_omnixas' / 'train_m3gnet_xas_pipeline.py').is_file():
    REPO_ROOT = REPO_ROOT.parent

SEED = 42
HEAD_PARALLEL = 4
ENCODER_NUM_WORKERS = 64
ENCODER_EPOCHS = 1000
ENCODER_LR = 1e-3
ROWS_PER_ELEMENT = 12
ENCODER_EVAL_BATCH = 512
HEAD_EPOCHS = 800
TUNED_EPOCHS = 1000
HEAD_PATIENCE = 60
HEAD_BATCH = 4096
OUTPUT_ROOT = REPO_ROOT / 'output/training/m3gnet_sweep'
RUN_NAME = f'm3gnet_sweep_seed{SEED}'
raw_root = Path(os.environ.get('OMNIXAS_DATA_ROOT', REPO_ROOT.parent / 'OmniXAS_data')) / 'materialscloud_omnixas_raw' / 'extracted'

print({
    'REPO_ROOT': str(REPO_ROOT),
    'raw_root': str(raw_root),
    'SEED': SEED,
    'HEAD_PARALLEL': HEAD_PARALLEL,
    'ENCODER_NUM_WORKERS': ENCODER_NUM_WORKERS,
    'ENCODER_EPOCHS': ENCODER_EPOCHS,
    'ENCODER_LR': ENCODER_LR,
    'ROWS_PER_ELEMENT': ROWS_PER_ELEMENT,
    'ENCODER_EVAL_BATCH': ENCODER_EVAL_BATCH,
    'HEAD_EPOCHS': HEAD_EPOCHS,
    'TUNED_EPOCHS': TUNED_EPOCHS,
    'HEAD_PATIENCE': HEAD_PATIENCE,
    'HEAD_BATCH': HEAD_BATCH,
    'OUTPUT_ROOT': str(OUTPUT_ROOT),
    'RUN_NAME': RUN_NAME,
})

In [ ]:
import json
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.expand_frame_repr', False)
import torch

import sweep_m3gnet_pipeline_lib as swp

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'device: {device}')
print(f'torch: {torch.__version__}')
torch.set_float32_matmul_precision('high')

In [ ]:
swp.encoder_preflight(REPO_ROOT, raw_root)
train_rows = {
    task: len(np.atleast_2d(np.loadtxt(REPO_ROOT / 'tutorial_omnixas' / 'ml_data' / f'{task}_train_y.txt', dtype=np.float32)))
    for task in swp.FEFF_TASKS
}
print('train rows per task:', train_rows)
print('total train rows:', sum(train_rows.values()))
for hidden_dims in ((500, 500, 550), (256, 256)):
    head = swp.SpectralHead(input_dim=64, hidden_dims=hidden_dims)
    print(f'SpectralHead {hidden_dims}: {sum(p.numel() for p in head.parameters()):,} parameters')

## Stage 1: 4 encoders (sequential)

In [ ]:
encoder_summary = []
for name, spec in swp.ENCODER_SPECS.items():
    run_dir = OUTPUT_ROOT / RUN_NAME / name
    run_dir.mkdir(parents=True, exist_ok=True)
    (run_dir / 'provenance.json').write_text(json.dumps({
        'pipeline': 'm3gnet_sweep',
        'git_sha': swp.git_sha(),
        'date': time.strftime('%Y-%m-%dT%H:%M:%S'),
        'spec': spec,
        'seed': SEED,
        'epochs': ENCODER_EPOCHS,
        'rows_per_element': ROWS_PER_ELEMENT,
        'precision': 'bf16-mixed if cuda else 32-true',
    }, indent=2), encoding='utf-8')
    start = time.time()
    ckpt = swp.train_encoder(
        name, spec, run_dir, REPO_ROOT, raw_root, SEED,
        epochs=ENCODER_EPOCHS, rows_per_element=ROWS_PER_ELEMENT,
        eval_batch=ENCODER_EVAL_BATCH, num_workers=ENCODER_NUM_WORKERS, lr=ENCODER_LR,
    )
    swp.export_features(ckpt, run_dir, REPO_ROOT, raw_root, spec, eval_batch=ENCODER_EVAL_BATCH)
    wall = time.time() - start
    rows = {
        task: len(np.atleast_2d(np.loadtxt(REPO_ROOT / 'tutorial_omnixas' / 'ml_data' / f'{task}_train_y.txt', dtype=np.float32)))
        for task in swp.FEFF_TASKS
    }
    encoder_summary.append({
        'enc': name,
        'blocks': spec['blocks'],
        'feature_dim': spec['feature_dim'],
        'cutoff': spec['cutoff'],
        'threebody_cutoff': spec['threebody_cutoff'],
        'encoder_dropout': spec['dropout'],
        'wall_seconds': round(wall, 1),
        'train_rows': sum(rows.values()),
    })
    print(f'{name}: {wall / 60:.1f} min', flush=True)
encoder_table = pd.DataFrame(encoder_summary)
encoder_table

## Stage 2: 8 UniversalXAS per encoder (parallel head workers)

In [ ]:
enc_best_universal = {}
all_rows = []
for name, enc_spec in swp.ENCODER_SPECS.items():
    run_dir = OUTPUT_ROOT / RUN_NAME / name
    features = run_dir / 'features'
    jobs = []
    for variant in swp.UNIVERSAL_SPECS:
        spec = dict(swp.UNIVERSAL_SPECS[variant])
        spec['input_dim'] = enc_spec['feature_dim']
        jobs.append(
            dict(
                run_dir=str(run_dir),
                features_dir=str(features),
                repo_root=str(REPO_ROOT),
                stage='universal',
                enc=name,
                variant=variant,
                task=None,
                seed=SEED,
                epochs=HEAD_EPOCHS,
                patience=HEAD_PATIENCE,
                batch_size=HEAD_BATCH,
                spec=spec,
                source_state_path=None,
                log_path=str(run_dir / 'logs' / f'universal_{variant}.log'),
                torch_threads=max(1, os.cpu_count() // HEAD_PARALLEL),
            )
        )
    results = swp.run_jobs_parallel(jobs, HEAD_PARALLEL)
    if len(results) != len(jobs):
        raise RuntimeError('Universal stage did not return all results')
    rows = []
    for result in results:
        row = {'enc': name, 'variant': result['variant'], 'macro_val_eta': result['macro_val_eta'], 'best_epoch': result['best_epoch']}
        for task in swp.FEFF_TASKS:
            row[f'val_eta_{task}'] = result['val_metrics'][task]['val_eta']
        rows.append(row)
    table = pd.DataFrame(rows)
    table.to_csv(run_dir / 'universal_selection.csv', index=False)
    best_row = table.loc[table['macro_val_eta'].idxmax()]
    best_variant = best_row['variant']
    best_macro = float(best_row['macro_val_eta'])
    enc_best_universal[name] = best_variant
    all_rows.extend(rows)
    print(f'{name}: best universal = {best_variant} (macro val eta {best_macro:.3f})', flush=True)
universal_selection_all = pd.DataFrame(all_rows)
universal_selection_all.round(3)


## Stage 3: 16 tuned variants x 8 elements from each encoder's best universal (parallel)

In [ ]:
tuned_all_rows = []
tuned_winners = {}
for name, enc_spec in swp.ENCODER_SPECS.items():
    run_dir = OUTPUT_ROOT / RUN_NAME / name
    features = run_dir / 'features'
    best_variant = enc_best_universal[name]
    source = run_dir / 'heads' / 'universal' / best_variant / 'best.pt'
    if not source.is_file():
        raise FileNotFoundError(f'Missing best universal checkpoint: {source}')
    source_hidden_dims = tuple(swp.UNIVERSAL_SPECS[best_variant]['hidden_dims'])
    jobs = []
    for variant, variant_spec in swp.TUNED_SPECS.items():
        spec = dict(variant_spec)
        spec['input_dim'] = enc_spec['feature_dim']
        spec['hidden_dims'] = list(source_hidden_dims)
        for task in swp.FEFF_TASKS:
            jobs.append(dict(
                run_dir=str(run_dir),
                features_dir=str(features),
                repo_root=str(REPO_ROOT),
                stage='tuned',
                enc=name,
                variant=variant,
                task=task,
                seed=SEED,
                epochs=TUNED_EPOCHS,
                patience=HEAD_PATIENCE,
                batch_size=HEAD_BATCH,
                spec=spec,
                source_state_path=str(source),
                log_path=str(run_dir / 'logs' / f'tuned_{variant}_{task}.log'),
                torch_threads=max(1, os.cpu_count() // HEAD_PARALLEL),
            ))
    results = swp.run_jobs_parallel(jobs, HEAD_PARALLEL)
    if len(results) != len(jobs):
        raise RuntimeError('Tuned stage did not return all results')
    for result in results:
        metrics = result['val_metrics'][result['task']]
        tuned_all_rows.append({
            'enc': name,
            'variant': result['variant'],
            'task': result['task'],
            'val_eta': metrics['val_eta'],
            'val_median_mse': metrics['val_median_mse'],
            'val_mse': metrics['val_mse'],
            'best_epoch': result['best_epoch'],
            'epochs_run': result['epochs_run'],
        })
    table = pd.DataFrame(tuned_all_rows)
    encoder_rows = table[table['enc'] == name]
    encoder_rows.to_csv(run_dir / 'tuned_validation.csv', index=False)
    for task in swp.FEFF_TASKS:
        sub = encoder_rows[encoder_rows['task'] == task]
        winner_row = sub.loc[sub['val_eta'].idxmax()]
        tuned_winners[(name, task)] = (winner_row['variant'], winner_row['val_eta'])
    print(f'{name}: {len(results)} tuned heads complete', flush=True)
tuned_rows_all = []
for name in swp.ENCODER_SPECS:
    for task in swp.FEFF_TASKS:
        variant, eta = tuned_winners[(name, task)]
        tuned_rows_all.append({'enc': name, 'element': task.removesuffix('_FEFF'), 'variant': variant, 'val_eta': eta})
tuned_winners_all = pd.DataFrame(tuned_rows_all)
tuned_winners_all.round(3)


In [ ]:
# Test split is touched only here, after all selection.
test_rows = []
for name, enc_spec in swp.ENCODER_SPECS.items():
    run_dir = OUTPUT_ROOT / RUN_NAME / name
    features = run_dir / 'features'
    splits = {task: swp._load_splits(features, task) for task in swp.FEFF_TASKS}
    best_variant = enc_best_universal[name]
    universal_spec = swp.UNIVERSAL_SPECS[best_variant]
    universal_head = swp.SpectralHead(
        input_dim=enc_spec['feature_dim'],
        hidden_dims=tuple(universal_spec['hidden_dims']),
        dropout=universal_spec['dropout'],
    )
    state = torch.load(run_dir / 'heads' / 'universal' / best_variant / 'best.pt', map_location='cpu', weights_only=False)
    universal_head.load_state_dict(state['state_dict'], strict=True)
    universal_head.to(device)
    eval_pairs = {task: (splits[task]['test'][0], splits[task]['test'][1], splits[task]['train'][1]) for task in swp.FEFF_TASKS}
    universal_test = swp.per_element_val_metrics(universal_head, eval_pairs, device, 4096)
    encoder_rows = []
    for task in swp.FEFF_TASKS:
        m = universal_test[task]
        encoder_rows.append({
            'enc': name, 'kind': 'universal', 'variant': best_variant, 'task': task,
            'test_eta': m['val_eta'], 'test_mse': m['val_mse'], 'test_median_mse': m['val_median_mse'],
        })
    for task in swp.FEFF_TASKS:
        winner_variant = tuned_winners[(name, task)][0]
        tuned_head = swp.SpectralHead(
            input_dim=enc_spec['feature_dim'],
            hidden_dims=tuple(universal_spec['hidden_dims']),
            dropout=universal_spec['dropout'],
        )
        tuned_state = torch.load(run_dir / 'heads' / 'tuned' / winner_variant / task / 'best.pt', map_location='cpu', weights_only=False)
        tuned_head.load_state_dict(tuned_state['state_dict'], strict=True)
        tuned_head.to(device)
        m = swp.task_val_metrics(tuned_head, splits[task]['test'][0], splits[task]['test'][1], splits[task]['train'][1], device, 4096)
        encoder_rows.append({
            'enc': name, 'kind': 'tuned', 'variant': winner_variant, 'task': task,
            'test_eta': m['val_eta'], 'test_mse': m['val_mse'], 'test_median_mse': m['val_median_mse'],
        })
    frame = pd.DataFrame(encoder_rows)
    frame.to_csv(run_dir / 'test_selected.csv', index=False)
    test_rows.extend(encoder_rows)
    print(f'{name}: test evaluated once for {len(frame)} selected heads')
combined = pd.DataFrame(test_rows)
combined.to_csv(OUTPUT_ROOT / RUN_NAME / 'test_selected_all.csv', index=False)
combined.round(3)


In [ ]:
universal_rows = []
for name in swp.ENCODER_SPECS:
    table = pd.read_csv(OUTPUT_ROOT / RUN_NAME / name / 'universal_selection.csv')
    eta_cols = [f'val_eta_{task}' for task in swp.FEFF_TASKS]
    best_row = table.loc[table['macro_val_eta'].idxmax()]
    universal_rows.append({
        'enc': name,
        'best_universal_variant': best_row['variant'],
        'macro_val_eta': best_row['macro_val_eta'],
        'mean_per_element_val_eta': best_row[eta_cols].mean(),
    })
enc_universal_table = pd.DataFrame(universal_rows)

tuned_rows = []
for name in swp.ENCODER_SPECS:
    for task in swp.FEFF_TASKS:
        variant, eta = tuned_winners[(name, task)]
        tuned_rows.append({'enc': name, 'task': task, 'best_tuned_variant': variant, 'val_eta': eta})
tuned_winners_table = pd.DataFrame(tuned_rows)

overall = tuned_winners_table.groupby('enc')['val_eta'].mean().sort_values(ascending=False)
overall_winner = overall.idxmax()

best_rows = []
for name in swp.ENCODER_SPECS:
    u_table = pd.read_csv(OUTPUT_ROOT / RUN_NAME / name / 'universal_selection.csv')
    best_row = u_table.loc[u_table['macro_val_eta'].idxmax()]
    tsel = pd.read_csv(OUTPUT_ROOT / RUN_NAME / name / 'test_selected.csv')
    for task in swp.FEFF_TASKS:
        u_test = tsel[(tsel['kind'] == 'universal') & (tsel['task'] == task)]['test_eta'].iloc[0]
        t_variant, t_val = tuned_winners[(name, task)]
        t_test = tsel[(tsel['kind'] == 'tuned') & (tsel['task'] == task)]['test_eta'].iloc[0]
        best_rows.append({
            'enc': name,
            'element': task.removesuffix('_FEFF'),
            'universal_variant': best_row['variant'],
            'universal_val_eta': float(best_row[f'val_eta_{task}']),
            'universal_test_eta': float(u_test),
            'tuned_variant': t_variant,
            'tuned_val_eta': float(t_val),
            'tuned_test_eta': float(t_test),
            'tuned_minus_universal_test': float(t_test - u_test),
        })
best_results = pd.DataFrame(best_rows)
best_results.to_csv(OUTPUT_ROOT / RUN_NAME / 'best_results.csv', index=False)

summary = {
    'git_sha': swp.git_sha(),
    'timestamp': time.strftime('%Y-%m-%dT%H:%M:%S'),
    'seed': SEED,
    'selection_note': 'Selection uses validation only. Test evaluated exactly once after selection: best universal per encoder and per-element tuned winners.',
    'encoder_summary': encoder_summary,
    'encoder_universal_table': enc_universal_table.to_dict('records'),
    'tuned_winners_table': tuned_winners_table.to_dict('records'),
    'overall_winner': overall_winner,
    'overall_mean_tuned_winner_val_eta': float(overall.max()),
    'best_results': best_rows,
}
(OUTPUT_ROOT / RUN_NAME / 'sweep_summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
display(enc_universal_table.round(3))
print('mean val eta of the 8 tuned winners per encoder:')
display(overall.round(3))
print(f'Overall winner: {overall_winner} (mean tuned-winner val eta {overall.max():.3f})')
best_results.round(3)


## Best per element
For each of the 8 elements this picks the best configuration by VALIDATION eta across the 8 candidates per element (4 encoder-best universals + 4 per-element tuned winners, from `best_results.csv`). Settings are read from each head's `metrics.json` sidecar (provenance). Test eta is reported for the chosen configuration only. No selection uses test.

In [ ]:
# Best configuration per element: argmax validation eta over the 8 candidates
# per element (4 encoder-best universals + 4 per-element tuned winners) in
# best_results.csv. Test eta is reported for the chosen configuration only.
best_res = pd.read_csv(OUTPUT_ROOT / RUN_NAME / 'best_results.csv')
cands = []
for row in best_res.itertuples(index=False):
    task = f"{row.element}_FEFF"
    cands.append(dict(element=task, kind='universal', enc=row.enc, variant=row.universal_variant,
                      val_eta=float(row.universal_val_eta), test_eta=float(row.universal_test_eta)))
    cands.append(dict(element=task, kind='tuned', enc=row.enc, variant=row.tuned_variant,
                      val_eta=float(row.tuned_val_eta), test_eta=float(row.tuned_test_eta)))
cand_df = pd.DataFrame(cands)
bpe_rows = []
for task in swp.FEFF_TASKS:
    sub = cand_df[cand_df['element'] == task]
    pick = sub.loc[sub['val_eta'].idxmax()]
    if pick['kind'] == 'universal':
        sidecar = OUTPUT_ROOT / RUN_NAME / pick['enc'] / 'heads' / 'universal' / pick['variant'] / 'metrics.json'
    else:
        sidecar = OUTPUT_ROOT / RUN_NAME / pick['enc'] / 'heads' / 'tuned' / pick['variant'] / task / 'metrics.json'
    if not sidecar.is_file():
        raise FileNotFoundError(
            f"Missing metrics.json sidecar for selected {pick['kind']} head {pick['variant']} ({task}): {sidecar}")
    s = json.loads(sidecar.read_text(encoding='utf-8'))['spec']
    e = swp.ENCODER_SPECS[pick['enc']]
    enc_settings = f"{e['blocks']} blocks, {e['feature_dim']}D, cutoff {e['cutoff']:g}, gnn do {e['dropout']:g}"
    opt = str(s.get('optimizer', 'adam'))
    if opt == 'adamw' and s.get('weight_decay') is not None:
        opt = f"adamw wd {s['weight_decay']:g}"
    if 'lr' in s:
        lr_str = f"{s['lr']:g}"
    else:
        lr_str = f"ph1 {s['lr_phase1']:g} -> ph2 {s['lr_phase2']:g}"
    sched = str(s.get('schedule', 'none'))
    extras = []
    if s.get('cosine_t') is not None:
        extras.append(f"T{int(s['cosine_t'])}")
    if s.get('warmup_epochs') is not None:
        extras.append(f"W{int(s['warmup_epochs'])}")
    if s.get('freeze_epochs') is not None:
        extras.append(f"F{int(s['freeze_epochs'])}")
    if s.get('div_factor') is not None:
        extras.append(f"D{s['div_factor']:g}")
    if extras:
        sched = f"{sched} ({' '.join(extras)})"
    parts = [
        f"hidden {'/'.join(map(str, s['hidden_dims']))}",
        f"do {s.get('head_dropout', s.get('dropout', 0.10)):g}",
        f"{opt} lr {lr_str}",
        sched,
        f"ES {s['es_metric']}",
        f"batch {s.get('batch_size', 4096)}",
        f"patience {s.get('patience', 60)}",
    ]
    if s.get('deriv_lambda'):
        parts.append(f"deriv {s['deriv_lambda']:g}")
    bpe_rows.append({
        'element': task.removesuffix('_FEFF'),
        'best_kind': pick['kind'],
        'encoder': pick['enc'],
        'variant': pick['variant'],
        'val_eta': float(pick['val_eta']),
        'test_eta': float(pick['test_eta']),
        'encoder_settings': enc_settings,
        'variant_settings': '; '.join(parts),
    })
best_per_element = pd.DataFrame(bpe_rows)
best_per_element.to_csv(OUTPUT_ROOT / RUN_NAME / 'best_per_element.csv', index=False)
print('Best per element (selected by val eta; test reported for the choice only):', flush=True)
best_per_element.set_index('element').round(3)


In [ ]:
# (a) macro val eta of the 8 universal variants, per encoder
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
for ax, name in zip(axes.ravel(), swp.ENCODER_SPECS):
    table = pd.read_csv(OUTPUT_ROOT / RUN_NAME / name / 'universal_selection.csv')
    x = np.arange(len(table))
    ax.bar(x, table['macro_val_eta'], width=0.8)
    ax.set_xticks(x, table['variant'], rotation=45, fontsize=8)
    ax.set_title(name)
    ax.set_ylabel('macro val eta')
plt.tight_layout()
fig.savefig(OUTPUT_ROOT / RUN_NAME / 'universal_macro_val_eta.png', dpi=120)

# (b) tuned winner vs best universal per element, per encoder
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
for ax, name in zip(axes.ravel(), swp.ENCODER_SPECS):
    table = pd.read_csv(OUTPUT_ROOT / RUN_NAME / name / 'universal_selection.csv')
    best_variant = enc_best_universal[name]
    best_row = table.loc[table['variant'] == best_variant].iloc[0]
    u_eta = [float(best_row[f'val_eta_{task}']) for task in swp.FEFF_TASKS]
    t_eta = [tuned_winners[(name, task)][1] for task in swp.FEFF_TASKS]
    x = np.arange(len(swp.FEFF_TASKS))
    width = 0.38
    ax.bar(x - width / 2, u_eta, width, label=f'universal {best_variant}')
    ax.bar(x + width / 2, t_eta, width, label='tuned winner')
    ax.set_xticks(x, [task.removesuffix('_FEFF') for task in swp.FEFF_TASKS], rotation=45)
    ax.set_title(name)
    ax.set_ylabel('val eta')
    ax.legend(fontsize=8)
plt.tight_layout()
fig.savefig(OUTPUT_ROOT / RUN_NAME / 'tuned_vs_universal_val_eta.png', dpi=120)

# (c) tuned val eta by variant for the overall winning encoder, colored by element
frame = pd.read_csv(OUTPUT_ROOT / RUN_NAME / overall_winner / 'tuned_validation.csv')
variant_index = {v: i for i, v in enumerate(swp.TUNED_SPECS)}
fig, ax = plt.subplots(figsize=(10, 5))
for task in swp.FEFF_TASKS:
    sub = frame[frame['task'] == task]
    ax.scatter(sub['variant'].map(variant_index), sub['val_eta'], label=task.removesuffix('_FEFF'), alpha=0.8)
ax.set_xticks(list(range(len(swp.TUNED_SPECS))), list(swp.TUNED_SPECS), rotation=90, fontsize=7)
ax.set_xlabel('tuned variant')
ax.set_ylabel('val eta')
ax.set_title(f'tuned val eta by variant: {overall_winner} (128 runs)')
ax.legend(fontsize=7)
plt.tight_layout()
fig.savefig(OUTPUT_ROOT / RUN_NAME / overall_winner / 'tuned_val_eta_scatter.png', dpi=120)
fig.savefig(OUTPUT_ROOT / RUN_NAME / f'tuned_val_eta_scatter_{overall_winner}.png', dpi=120)
print('Plots saved to the sweep root and the winning encoder run dir.')

## How to read results / caveats

- Selection uses validation only. Test is reported exactly once, in the test
  evaluation cell, after all selection.
- Seed 42 is fixed for every variant on purpose: variant deltas attribute the
  hyperparameters, not seed variance.
- `enc_c_wide128` doubles the feature width (128D features); universal and
  tuned head `input_dim` follows the encoder feature dimension.
- Encoder stage trains the encoder jointly with a full 500/500/550
  SpectralHead at the 1000x feature scale, mirroring the pipeline
  `LitScratch`; the head is discarded and only encoder weights are saved
  to `best_encoder.ckpt`.
- Resume semantics: rerunning the notebook reuses existing checkpoints
  (encoders and heads) and deletes nothing.
- `run_jobs_parallel` uses the `spawn` start method (CUDA-safe), so all head
  job logic lives in `sweep_m3gnet_pipeline_lib.py`; job dicts contain only
  picklable strings, numbers, and dicts.
- `ENCODER_NUM_WORKERS` for graph construction may need tuning on the target
  box (file descriptor and RAM limits).
- Total jobs: 4 encoders + 32 universal heads + 512 tuned heads.
- Tables render as HTML tables in the notebook; rounding is display-only, CSVs keep raw values. `best_results.csv` at the sweep root is the combined best-results table (32 rows: one per encoder x element, val and test eta of the selected universal and tuned heads). `best_per_element.csv` adds the single best configuration per element with its settings.
